# Ôn tập Buổi 09 - Seaborn và EDA end-to-end

        **Thời lượng gợi ý:** 75-90 phút  
        **Cách học:** trả lời câu hỏi trước khi chạy cell; sau mỗi ví dụ, tự nói thành lời *đầu vào - phép biến đổi - đầu ra*.

        ## Mục tiêu

        - Chọn đúng họ biểu đồ Seaborn.
- Thực hiện EDA theo Load -> Inspect -> Quality -> Clean -> Analyze -> Visualize -> Interpret.
- Viết kết luận có bằng chứng, giới hạn và không nhầm tương quan với nhân quả.

        > Notebook này là tài liệu ôn chủ động, không thay thế toàn bộ slide. Khi một câu tự kiểm tra chưa chắc, quay lại đúng mục tương ứng trong `slides/buoi9_python_datascience.pdf`.


In [ ]:
from pathlib import Path

HERE = Path.cwd().resolve()
ROOT = next(
    (p for p in (HERE, *HERE.parents) if (p / "datasets").exists() and (p / "slides").exists()),
    None,
)
assert ROOT is not None, "Hãy mở notebook từ bên trong repo Hoan-Data-Science-Course."
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
DATA_DIR = ROOT / 'datasets' / 'buoi6'
sns.set_theme(style='whitegrid')
print(f"Seaborn: {sns.__version__}")
print(f'Repo: {ROOT}')


## 0. Chẩn đoán nhanh - chưa chạy code

**1. `pairplot` phù hợp khi nào?**

<details><summary>Kiểm tra đáp án</summary>

Khi muốn quét nhanh quan hệ từng cặp biến số; không phù hợp nếu có quá nhiều biến hoặc quá nhiều điểm.

</details>

**2. Boxplot và violinplot khác nhau ở đâu?**

<details><summary>Kiểm tra đáp án</summary>

Boxplot tóm tắt median/quartile/outlier; violinplot cho thấy thêm hình dạng phân phối.

</details>

**3. EDA có chứng minh nhân quả không?**

<details><summary>Kiểm tra đáp án</summary>

Không. EDA mô tả pattern/association; nhân quả cần thiết kế nghiên cứu và giả định mạnh hơn.

</details>


## 1. Bốn họ biểu đồ

| Câu hỏi | Họ biểu đồ | Hàm gợi ý |
|---|---|---|
| Phân phối ra sao? | Distribution | `histplot`, `kdeplot` |
| X liên hệ Y thế nào? | Relational | `scatterplot`, `lineplot` |
| Các nhóm khác nhau ra sao? | Categorical | `boxplot`, `violinplot`, `countplot` |
| Nhiều biến/nhóm cùng lúc? | Multi-variable | `pairplot`, `jointplot`, `FacetGrid` |


## 2. LOAD + INSPECT + QUALITY


In [ ]:
titanic = pd.read_csv(DATA_DIR / "titanic.csv")
display(titanic.head())
quality = pd.DataFrame({
    "dtype": titanic.dtypes.astype(str),
    "missing_count": titanic.isna().sum(),
    "missing_rate": titanic.isna().mean(),
    "n_unique": titanic.nunique(dropna=True),
}).sort_values("missing_rate", ascending=False)
display(quality)
print("shape:", titanic.shape, "| duplicate rows:", titanic.duplicated().sum())
assert titanic.shape == (891, 15)


## 3. CLEAN + TRANSFORM

Không điền `deck`: tỷ lệ thiếu quá cao và cột không cần cho các câu hỏi bên dưới. Với `age`, giữ missing và để từng phân tích dùng tập con phù hợp. Dù có các hàng giống hệt nhau, file không có `passenger_id`, nên không đủ căn cứ để coi đó là cùng một hành khách và xóa bằng `drop_duplicates()`.


In [ ]:
analysis = (
    titanic.copy()
    .assign(
        age_group=lambda d: pd.cut(d["age"], [0, 12, 18, 35, 60, np.inf], labels=["Child", "Teen", "Young adult", "Adult", "Senior"]),
        family_size=lambda d: d["sibsp"] + d["parch"] + 1,
    )
)
assert analysis["family_size"].ge(1).all()


## 4. Distribution + Categorical


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=analysis, x="age", hue="survived", multiple="stack", bins=20, ax=axes[0])
sns.boxplot(data=analysis, x="class", y="fare", hue="survived", ax=axes[1])
axes[0].set_title("Phân phối tuổi theo sống sót")
axes[1].set_title("Giá vé theo hạng và sống sót")
axes[1].set_yscale("log")
fig.tight_layout()
plt.show()


## 5. Relational + Facet


In [ ]:
grid = sns.relplot(
    data=analysis, x="age", y="fare", hue="survived", col="sex",
    size="family_size", sizes=(15, 100), alpha=0.55, height=3.5,
)
grid.set_axis_labels("Tuổi", "Giá vé")
grid.set_titles("Giới tính: {col_name}")
plt.show()


## 6. ANALYZE: tỷ lệ luôn đi cùng cỡ mẫu


In [ ]:
evidence = (
    analysis.groupby(["sex", "class"], observed=True)
    .agg(n=("survived", "size"), survival_rate=("survived", "mean"), median_age=("age", "median"))
    .reset_index()
)
display(evidence.style.format({"survival_rate": "{:.1%}", "median_age": "{:.1f}"}))
assert evidence["n"].sum() == len(analysis)


## 7. INTERPRET - mẫu kết luận có kỷ luật

- **Quan sát:** tỷ lệ sống khác nhau theo giới tính và hạng vé trong dữ liệu Titanic.
- **Bằng chứng:** dùng bảng `n` + `survival_rate` và biểu đồ phía trên.
- **Giới hạn:** đây là dữ liệu quan sát lịch sử; nhóm có nhiều yếu tố gây nhiễu và missing age/deck.
- **Không được suy diễn:** không kết luận giới tính hay hạng vé *gây ra* sống sót chỉ từ EDA.


## 8. Tự kiểm tra tích hợp


In [ ]:
checks = {
    "raw_rows_preserved_without_identifier": len(analysis) == len(titanic),
    "survival_probability_valid": evidence["survival_rate"].between(0, 1).all(),
    "all_rows_accounted_for": evidence["n"].sum() == len(analysis),
    "derived_family_size_valid": analysis["family_size"].ge(1).all(),
}
print(checks)
assert all(checks.values())


## Bài tự luyện

        Chọn một câu hỏi mới trên Titanic. Viết đủ 5 dòng: câu hỏi, cột dùng, kiểm tra chất lượng, bảng thống kê, biểu đồ và kết luận có giới hạn.

        <details><summary>Gợi ý / đáp án tham khảo</summary>

        ```python
        question = "Đi một mình có liên hệ với tỷ lệ sống không?"
table = analysis.groupby("alone").agg(n=("survived", "size"), survival_rate=("survived", "mean"))
display(table)
sns.barplot(data=analysis, x="alone", y="survived", errorbar=("ci", 95))
plt.ylabel("Tỷ lệ sống ước tính")
plt.show()
# Kết luận phải đề cập n, chênh lệch quan sát và giới hạn dữ liệu quan sát.
        ```

        </details>


## Phiếu rời buổi

        Không nhìn lại notebook, hãy tự xác nhận:

        - [ ] Tôi chọn biểu đồ dựa trên loại câu hỏi và biến.
- [ ] Tôi thực hiện đủ Load -> Inspect -> Quality -> Clean -> Analyze -> Visualize -> Interpret.
- [ ] Mọi kết luận của tôi có bằng chứng, cỡ mẫu và giới hạn.

        Nếu chưa đánh dấu được một mục, ghi lại **một ví dụ do chính bạn nghĩ ra** rồi chạy thử.
